# Option 1: Double Machine Learning (DML)
**Replication**: Chernozhukov et al. (2018) - Double/Debiased Machine Learning
**Data**: 401(k) Eligibility and Net Wealth (Poterba, Venti & Wise, 1994)

**Key Question**: Does 401(k) eligibility increase net wealth?

## 1. Setup and Data Loading

In [ ]:
# Download data files if not already present
import os
import urllib.request

BASE_URL = "https://raw.githubusercontent.com/JasmineHao/JasmineHao.github.io/main/econ6083/final-project/notebooks/data/"
DATA_FILES = ['dml_401k.csv']

os.makedirs('data', exist_ok=True)
for fname in DATA_FILES:
    if not os.path.exists(f'data/{fname}'):
        print(f"Downloading {fname} ...")
        urllib.request.urlretrieve(BASE_URL + fname, f'data/{fname}')
        print(f"  Saved to data/{fname}")
    else:
        print(f"Found local: data/{fname}")


In [ ]:
# Install required packages
!pip install -q doubleml pandas numpy scikit-learn matplotlib statsmodels

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.linear_model import LassoCV, LogisticRegressionCV, RidgeCV
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

# Load 401k data from local file
# Same data used in Chernozhukov et al. (2018)
df = pd.read_csv('data/dml_401k.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()

## 2. Data Exploration

In [ ]:
# Key variables
# net_tfa: Net financial assets (outcome)
# e401: 401(k) eligibility (treatment)
# covariates: age, inc, educ, fsize, marr, twoearn, db, pira, hown

print("Outcome: Net Financial Assets")
print(df['net_tfa'].describe())

print("\nTreatment: 401(k) Eligibility")
print(df['e401'].value_counts())

print("\nRaw Difference in Means:")
print(df.groupby('e401')['net_tfa'].mean())
raw_ate = df[df['e401']==1]['net_tfa'].mean() - df[df['e401']==0]['net_tfa'].mean()
print(f"\nRaw ATE estimate: {raw_ate:.2f}")

## 3. DML Implementation

In [ ]:
def dml_plr(Y, D, X, ml_g, ml_m, n_folds=5, binary_treatment=True, random_state=42):
    """
    Double Machine Learning for Partially Linear Regression
    
    Model: Y = theta*D + g(X) + eps
           D = m(X) + eta
    
    Returns sqrt(n)-consistent estimate of theta via Neyman orthogonality
    """
    n = len(Y)
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    
    # Store predictions
    Y_hat = np.zeros(n)
    D_hat = np.zeros(n)
    
    for train_idx, test_idx in kf.split(X):
        X_train, X_test = X[train_idx], X[test_idx]
        Y_train, Y_test = Y[train_idx], Y[test_idx]
        D_train, D_test = D[train_idx], D[test_idx]
        
        # Fit outcome model
        ml_g.fit(X_train, Y_train)
        Y_hat[test_idx] = ml_g.predict(X_test)
        
        # Fit propensity/treatment model
        if binary_treatment:
            ml_m.fit(X_train, D_train)
            D_hat[test_idx] = ml_m.predict_proba(X_test)[:, 1]
        else:
            ml_m.fit(X_train, D_train)
            D_hat[test_idx] = ml_m.predict(X_test)
    
    # Compute residuals
    Y_tilde = Y - Y_hat
    D_tilde = D - D_hat
    
    # DML estimator (orthogonal moment)
    theta = np.mean(Y_tilde * D_tilde) / np.mean(D_tilde ** 2)
    
    # Standard error (sandwich estimator)
    psi = Y_tilde - D_tilde * theta
    J = np.mean(D_tilde ** 2)
    var = np.mean(psi ** 2) / (J ** 2)
    se = np.sqrt(var / n)
    
    # 95% Confidence interval
    ci_lower = theta - 1.96 * se
    ci_upper = theta + 1.96 * se
    
    return {
        'theta': theta,
        'se': se,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'Y_hat': Y_hat,
        'D_hat': D_hat,
        'psi': psi
    }

## 4. Prepare Data for DML

In [ ]:
# Define variables
Y = df['net_tfa'].values
D = df['e401'].values

# Covariates (same as Chernozhukov et al. 2018)
covariates = ['age', 'inc', 'educ', 'fsize', 'marr', 'twoearn', 'db', 'pira', 'hown']
X = df[covariates].values

print(f"Outcome shape: {Y.shape}")
print(f"Treatment shape: {D.shape}")
print(f"Covariates shape: {X.shape}")
print(f"\nCovariates: {covariates}")

## 5. Run DML with Different ML Methods

Replicating Table 2 from Chernozhukov et al. (2018): ATE of 401(k) eligibility on net financial assets

In [ ]:
results = []

# 1. OLS (naive benchmark - no ML)
X_ols = sm.add_constant(np.column_stack([D, X]))
ols_model = sm.OLS(Y, X_ols).fit()
ols_coef = ols_model.params[1]
ols_se = ols_model.bse[1]
results.append(['OLS', ols_coef, ols_se, ols_coef/ols_se])

# 2. DML with Lasso
ml_g_lasso = LassoCV(cv=5, random_state=42)
ml_m_lasso = LogisticRegressionCV(cv=5, random_state=42, max_iter=1000)
dml_lasso = dml_plr(Y, D, X, ml_g_lasso, ml_m_lasso, n_folds=5)
results.append(['DML-Lasso', dml_lasso['theta'], dml_lasso['se'], 
                dml_lasso['theta']/dml_lasso['se']])

# 3. DML with Ridge
ml_g_ridge = RidgeCV(cv=5)
ml_m_ridge = LogisticRegressionCV(cv=5, random_state=42, max_iter=1000)
dml_ridge = dml_plr(Y, D, X, ml_g_ridge, ml_m_ridge, n_folds=5)
results.append(['DML-Ridge', dml_ridge['theta'], dml_ridge['se'], 
                dml_ridge['theta']/dml_ridge['se']])

# 4. DML with Random Forest
ml_g_rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
ml_m_rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
dml_rf = dml_plr(Y, D, X, ml_g_rf, ml_m_rf, n_folds=5)
results.append(['DML-RF', dml_rf['theta'], dml_rf['se'], 
                dml_rf['theta']/dml_rf['se']])

# Create results table
results_df = pd.DataFrame(results, columns=['Method', 'ATE', 'SE', 't-stat'])
results_df['CI_lower'] = results_df['ATE'] - 1.96 * results_df['SE']
results_df['CI_upper'] = results_df['ATE'] + 1.96 * results_df['SE']

print("\nDML Results: Effect of 401(k) Eligibility on Net Financial Assets\n")
print(results_df.round(3))

print("\n" + "="*60)
print("Key Finding:")
print(f"DML-Lasso ATE: {dml_lasso['theta']:.0f} (SE: {dml_lasso['se']:.0f})")
print(f"95% CI: [{dml_lasso['ci_lower']:.0f}, {dml_lasso['ci_upper']:.0f}]")
print("\nInterpretation: 401(k) eligibility increases net financial assets")
print("by approximately $8,000-$9,000 on average.")
print("="*60)

## 6. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Comparison of methods
ax1 = axes[0]
methods = results_df['Method']
ates = results_df['ATE']
ci_lower = results_df['CI_lower']
ci_upper = results_df['CI_upper']

ax1.errorbar(methods, ates, yerr=[ates-ci_lower, ci_upper-ates], 
             fmt='o', capsize=5, capthick=2, markersize=8)
ax1.axhline(y=0, color='red', linestyle='--', alpha=0.5)
ax1.set_ylabel('ATE ($)')
ax1.set_title('DML Estimates: Effect of 401(k) on Net Assets')
ax1.grid(alpha=0.3)
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45)

# Plot 2: Residuals check
ax2 = axes[1]
ax2.scatter(dml_lasso['D_hat'], dml_lasso['Y_hat'], alpha=0.3)
ax2.set_xlabel('Predicted Treatment (Propensity)')
ax2.set_ylabel('Predicted Outcome')
ax2.set_title('Nuisance Function Predictions')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Extension: Heterogeneous Effects by Income Quartile

Analyze how treatment effects vary by income level.

In [ ]:
# Create income quartiles
df['income_quartile'] = pd.qcut(df['inc'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

het_results = []
for q in ['Q1', 'Q2', 'Q3', 'Q4']:
    mask = df['income_quartile'] == q
    Y_q = Y[mask]
    D_q = D[mask]
    X_q = X[mask]
    
    dml_q = dml_plr(Y_q, D_q, X_q, ml_g_lasso, ml_m_lasso, n_folds=3)
    het_results.append([
        q, 
        dml_q['theta'], 
        dml_q['se'], 
        dml_q['ci_lower'], 
        dml_q['ci_upper']
    ])

het_df = pd.DataFrame(het_results, 
                      columns=['Quartile', 'ATE', 'SE', 'CI_lower', 'CI_upper'])

print("\nHeterogeneous Effects by Income Quartile:")
print(het_df.round(2))

# Plot
plt.figure(figsize=(10, 6))
plt.errorbar(het_df['Quartile'], het_df['ATE'], 
             yerr=[het_df['ATE']-het_df['CI_lower'], het_df['CI_upper']-het_df['ATE']], 
             fmt='o-', capsize=5, capthick=2, markersize=10)
plt.axhline(y=0, color='red', linestyle='--', alpha=0.5)
plt.xlabel('Income Quartile')
plt.ylabel('ATE ($)')
plt.title('Heterogeneous Treatment Effects by Income')
plt.grid(alpha=0.3)
plt.show()

## Interpreting Your Results

| Output | What it means | What to look for |
|---|---|---|
| **Naive OLS estimate** | Simple regression of Y on T, ignoring confounders | Likely biased upward if treated individuals are wealthier |
| **DML estimate** | Causal effect after partialling out confounders using ML | Should be smaller and more precise than OLS if confounders matter |
| **Standard error** | Uncertainty around the DML estimate | Compare across methods (Lasso vs Ridge vs RF) |
| **Heterogeneous effects** | CATE by income quartile | Do richer households benefit more from 401(k) access? |

**Key question**: Does DML change the conclusion compared to naive OLS? If so, why?

## Summary

This notebook replicates the main result from Chernozhukov et al. (2018):

1. **Raw difference**: ~$13,000 (biased due to selection)
2. **DML-Lasso ATE**: ~$8,000-$9,000 (robust to confounding)
3. **Significance**: Statistically significant at 1% level
4. **Heterogeneity**: Effects larger for higher income groups

**Extensions to try**:
- Try different ML methods (XGBoost, Neural Networks)
- Test sensitivity to number of folds
- Estimate effects on other outcomes (total wealth, non-financial assets)